# Apache Spark Structured Streaming — Combined Notebook

This notebook merges all topics from the original Databricks export (`ApacheSparkStreaming.dbc`) into a single sequential notebook.

**Sample data:** `day1.json`, `day2.json`, `day3.json` (place alongside this notebook, or update the paths in the read cells — original notebooks used DBFS/Volumes paths like `/Volumes/workspace/stream/...`).


## Table of Contents
1. [stream tutorial](#stream-tutorial)
2. [OutPut Modes](#output-modes)
3. [Windows](#windows)
4. [ForEachBatch](#foreachbatch)


---
## stream tutorial


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Read Streaming Data**

In [ ]:
my_schema = """order_id STRING,
        timestamp STRING,
        customer STRUCT<
          customer_id: INT,
          name: STRING,
          email: STRING,
          address: STRUCT<
            city: STRING,
            postal_code: STRING,
            country: STRING
          >
        >,
        items ARRAY<STRUCT<
          item_id: STRING,
          product_name: STRING,
          quantity: INT,
          price: DOUBLE
        >>,
        payment STRUCT<
          method: STRING,
          transaction_id: STRING
        >,
        metadata ARRAY<STRUCT<
          key: STRING,
          value: STRING
        >>"""

In [ ]:
df = spark.readStream.format("json")\
          .option("multiLine",True)\
          .schema(my_schema)\
          .load("/Volumes/workspace/stream/streaming/jsonsource")


# Transformations
df = df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

df = df.withColumn("items",explode_outer("items"))

df = df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","customer_id","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")

df = df.withColumn("metadata",explode_outer("metadata"))
df = df.select("*","metadata.key","metadata.value").drop("metadata")


In [ ]:
df.writeStream.format("delta")\
        .outputMode("append")\
        .trigger(once=True)\
        .option("path","/Volumes/workspace/stream/streaming/jsonsink/Data")\
        .option("checkpointLocation","/Volumes/workspace/stream/streaming/jsonsink/checkpoint")\
        .start()


{"text/plain": "<pyspark.sql.connect.streaming.query.StreamingQuery at 0xfff73fd34410>"}

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/workspace/stream/streaming/jsonsink/Data`

item_id | price | product_name | quantity | order_id | timestamp | customer_id | name | email | city | country | postal_code | method | transaction_id | key | value
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
I100 | 25.99 | Wireless Mouse | 2 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | campaign | back_to_school
I100 | 25.99 | Wireless Mouse | 2 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | channel | email
I101 | 15.49 | USB-C Adapter | 1 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | campaign | back_to_school
I101 | 15.49 | USB-C Adapter | 1 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | channel | email
I102 | 45.0 | Bluetooth Key

### **Archiving**

In [ ]:
dbutils.fs.mkdirs("/Volumes/workspace/stream/streaming/jsonsinknew")


{"text/plain": "True"}

In [ ]:
df = spark.readStream.format("json")\
        .option("multiLine",True)\
        .schema(my_schema)\
        .option("cleanSource","archive")\
        .option("sourceArchiveDir","/Volumes/workspace/stream/streaming/jsonsourcearchive")\
        .load("/Volumes/workspace/stream/streaming/jsonsourcenew")


# Transformations
df = df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

df = df.withColumn("items",explode_outer("items"))

df = df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","customer_id","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")

df = df.withColumn("metadata",explode_outer("metadata"))
df = df.select("*","metadata.key","metadata.value").drop("metadata")


In [ ]:
df.writeStream.format("delta")\
        .outputMode("append")\
        .trigger(once=True)\
        .option("path","/Volumes/workspace/stream/streaming/jsonsinknew/Data")\
        .option("checkpointLocation","/Volumes/workspace/stream/streaming/jsonsinknew/checkpoint")\
        .start()


{"text/plain": "<pyspark.sql.connect.streaming.query.StreamingQuery at 0xfff590fd3ad0>"}

### READ JSON DATA (BATCH)

In [ ]:
# df = spark.read.format("json")\
#           .option("inferShcema",True)\
#           .option("multiLine",True)\
#           .load("/Volumes/workspace/stream/streaming/jsonsource")

# display(df)

In [ ]:
# df = df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

# df = df.withColumn("items",explode_outer("items"))

# display(df)

items | order_id | timestamp | customer_id | name | email | city | country | postal_code | payment | metadata
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
['I100', 25.99, 'Wireless Mouse', 2] | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | ['Credit Card', 'TXN7890'] | [['campaign', 'back_to_school'], ['channel', 'email']]
['I101', 15.49, 'USB-C Adapter', 1] | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | ['Credit Card', 'TXN7890'] | [['campaign', 'back_to_school'], ['channel', 'email']]

In [ ]:
# df = df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","customer_id","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")
# display(df)

item_id | price | product_name | quantity | order_id | timestamp | customer_id | name | email | city | country | postal_code | method | transaction_id | metadata
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
I100 | 25.99 | Wireless Mouse | 2 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | [['campaign', 'back_to_school'], ['channel', 'email']]
I101 | 15.49 | USB-C Adapter | 1 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | [['campaign', 'back_to_school'], ['channel', 'email']]

In [ ]:
# df = df.withColumn("metadata",explode_outer("metadata"))
# df = df.select("*","metadata.key","metadata.value").drop("metadata")
# display(df)

item_id | price | product_name | quantity | order_id | timestamp | customer_id | name | email | city | country | postal_code | method | transaction_id | key | value
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
I100 | 25.99 | Wireless Mouse | 2 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | campaign | back_to_school
I100 | 25.99 | Wireless Mouse | 2 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | channel | email
I101 | 15.49 | USB-C Adapter | 1 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | campaign | back_to_school
I101 | 15.49 | USB-C Adapter | 1 | ORD1001 | 2025-06-01T10:15:00Z | 501 | John Doe | john@example.com | Toronto | Canada | M5H 2N2 | Credit Card | TXN7890 | channel | email

---
## OutPut Modes


In [ ]:
from delta.tables import DeltaTable

In [ ]:
DeltaTable.createIfNotExists(spark) \
  .tableName("workspace.stream.sourcetable") \
  .addColumn("color", "STRING")\
  .execute()

{"text/plain": "<delta.connect.tables.DeltaTable at 0xfff2a0324850>"}

In [ ]:
DeltaTable.createIfNotExists(spark) \
  .tableName("workspace.stream.sinktable") \
  .addColumn("color", "STRING")\
  .execute()

{"text/plain": "<delta.connect.tables.DeltaTable at 0xfff2a0466ed0>"}

In [ ]:
# %sql
INSERT INTO workspace.stream.sourcetable VALUES 
('maroon')

num_affected_rows | num_inserted_rows
--- | ---
1 | 1

In [ ]:
# %sql
SELECT * FROM workspace.stream.sourcetable

color
---
red
green
blue
yellow
orange
orange

In [ ]:
df = spark.readStream.table('workspace.stream.sourcetable')

In [ ]:
from pyspark.sql.functions import *

In [ ]:
df = df.groupBy("color").agg(count("*").alias("count"))

In [ ]:
df.writeStream.format("delta")\
            .outputMode('update')\
            .trigger(once=True)\
            .option("checkpointLocation","/Volumes/workspace/stream/streaming/output/check")\
            .option("path","/Volumes/workspace/stream/streaming/output/Data")\
            .start()

{"text/plain": "<pyspark.sql.connect.streaming.query.StreamingQuery at 0xfff21bb33210>"}

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/workspace/stream/streaming/output/Data`

color | count
--- | ---
yellow | 1
orange | 2
maroon | 1
green | 2
blue | 2
red | 2

---
## Windows


In [ ]:
# %sql
CREATE TABLE workspace.stream.windowtbl
(
  color STRING,
  event_date TIMESTAMP
)

In [ ]:
# %sql
INSERT INTO workspace.stream.windowtbl
VALUES
('red','2025-01-01T11:07:00.000+00:00')

num_affected_rows | num_inserted_rows
--- | ---
1 | 1

In [ ]:
df = spark.readStream.table("workspace.stream.windowtbl")

In [ ]:
from pyspark.sql.functions import * 

In [ ]:
df = df.groupBy("color",window("event_date","10 minutes"))\
            .agg(count(lit(1)).alias("color_count"))

In [ ]:
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(once=True)\
        .option("path","/Volumes/workspace/stream/streaming/windows/Data")\
        .option("checkpointLocation","/Volumes/workspace/stream/streaming/windows/checkpoint")\
        .start()

{"text/plain": "<pyspark.sql.connect.streaming.query.StreamingQuery at 0xfff5089d1750>"}

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/workspace/stream/streaming/windows/Data`

color | window | color_count
--- | --- | ---
green | ['2025-01-01T11:10:00.000Z', '2025-01-01T11:20:00.000Z'] | 1
green | ['2025-01-01T11:00:00.000Z', '2025-01-01T11:10:00.000Z'] | 2
red | ['2025-01-01T11:00:00.000Z', '2025-01-01T11:10:00.000Z'] | 2

---
## ForEachBatch


In [ ]:
from pyspark.sql.functions import *

In [ ]:
df = spark.readStream.table('workspace.stream.sourcetable')

In [ ]:
def myfunc(df,batch_id):

    df = df.groupBy("color").agg(count("*").alias("count"))

    # destination 1
    df.write.format("delta")\
            .mode("append")\
            .option("path","/Volumes/workspace/stream/streaming/foreachsink/dest1")\
            .save()

    # Destination 2 
    df.write.format("delta")\
            .mode("append")\
            .option("path","/Volumes/workspace/stream/streaming/foreachsink/dest2")\
            .save()

In [ ]:
df.writeStream.foreachBatch(myfunc)\
    .outputMode("append")\
    .trigger(once=True)\
    .option("checkpointLocation","/Volumes/workspace/stream/streaming/foreachsink/checkpoint")\
    .start()

{"text/plain": "<pyspark.sql.connect.streaming.query.StreamingQuery at 0xfffef359ca10>"}

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/workspace/stream/streaming/foreachsink/dest2`

color | count
--- | ---
green | 2
orange | 2
blue | 2
red | 2
yellow | 1
maroon | 1